In [1]:
!pip install langchain_huggingface langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is 

In [5]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [2]:
llm = HuggingFacePipeline.from_model_id(
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task = "text-generation"
)

model = ChatHuggingFace(llm=llm)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
# tool creation

@tool
def multiply(a: int,b: int) -> int:
  """Given two number this tool return their product"""
  return a*b

In [8]:
multiply.invoke({'a':4,'b':6})

24

In [10]:
llm_with_tools = model.bind_tools([multiply])

In [20]:
llm_with_tools.invoke('multiply 4 and 4')

AIMessage(content='<|user|>\nmultiply 4 and 4</s>\n<|assistant|>\nTo multiply 4 and 4 in a single operation in a Python program, you can use the `*` operator. For example:\n\n```python\n# multiplying 4 and 4\nprint(4 * 4)\n```\n\nThis will output `16`.\n\nThe `*` operator is an arithmetic operator that multiplies two numbers together. It returns the result of the operation.', additional_kwargs={}, response_metadata={}, id='lc_run--019daaa5-d091-77e0-9149-7d12861f5dcc-0', tool_calls=[], invalid_tool_calls=[])

### Currency Exchange Tool

In [34]:
# It is used when we are calling two functions parallely but second function needs the value of first function
# In a way we are saying: "LLM do not try to fill this argument", "I (the develpor/ runtime) will inject this value after running earlier tools"
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def currencyexchange(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base and target currency
  """
  url = f"https://v6.exchangerate-api.com/v6/<API key>/pair/{base_currency}/{target_currency}"

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """given a currency conversion rate this function calculate the target currency value from a given base currency value"""

  return base_currency_value * conversion_rate

In [35]:
currencyexchange.invoke({"base_currency":'PKR',"target_currency":'IRR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1776643201,
 'time_last_update_utc': 'Mon, 20 Apr 2026 00:00:01 +0000',
 'time_next_update_unix': 1776729601,
 'time_next_update_utc': 'Tue, 21 Apr 2026 00:00:01 +0000',
 'base_code': 'PKR',
 'target_code': 'IRR',
 'conversion_rate': 502.1234}

In [36]:
convert.invoke({'base_currency_value':100, 'conversion_rate':502.1234})

50212.340000000004

In [37]:
llm_with_tools = model.bind_tools([currencyexchange, convert])

In [38]:
message = [HumanMessage('What is the conversion factor between USD and PKR, and based on that convert 10 usd to pkr')]

In [39]:
llm_with_tools.invoke(message)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


AIMessage(content='<|user|>\nWhat is the conversion factor between USD and PKR, and based on that convert 10 usd to pkr</s>\n<|assistant|>\nThe conversion factor between USD (United States Dollar) and PKR (Pakistan Rupee) is 10.00, which means 1 USD is equivalent to 10 PKR.\n\nTo convert 10 USD to PKR, multiply the USD amount by the conversion factor (line above) to get the PKR amount:\n\nPKR amount = 10 USD × 10.00 = 1000 PKR\n\nTherefore, 10 USD is equivalent to 1000 PKR in Pakistani currency.', additional_kwargs={}, response_metadata={}, id='lc_run--019daac0-f227-7260-a00d-1160d9c73436-0', tool_calls=[], invalid_tool_calls=[])

### Our HuggingFace  LLM model is not working properly here but if we finetune it or use another LLM api such as OpenAI model then this code will work properly.